# Project 20 — Deep Eval (Capstone)

> **Goal:** score a RAG pipeline on generation quality — faithfulness, answer
> relevance, context precision, context recall — implemented from scratch in
> `evaluation/` and cross-checked against RAGAS 0.4.3. The judge
> (`qwen2.5-coder:7b` on Ollama) and the embeddings (`BAAI/bge-base-en-v1.5`
> via fastembed) run 100% locally: zero API keys, zero cost.

We evaluate the SD-08 invoice RAG pipeline (6 invoices, 23 hand-checked
question/answer pairs) end-to-end: retrieve → generate → score.


## 0 · Setup — environment & local models

**WHAT** — resolve the repo root (the notebook lives two levels below it),
check that Ollama is up with the judge model, import everything, and create
the two collaborators every metric needs: a local embedder and a local judge.

**WHY** — `evaluation/`, `vectordb/`, and `retrieval/` are namespace packages
with no `__init__.py`; the `sys.path` entry makes `from evaluation.golden
import ...` resolve. Everything after this section assumes the judge
(Ollama) is reachable.

**WHAT TO EXPECT** — repo paths print, Ollama's model list includes
`qwen2.5-coder:7b`, and the embedder reports 768-dimensional vectors.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent
DATA_DIR = REPO_ROOT / "Data" / "SD-08-invoices"

sys.path.insert(0, str(REPO_ROOT))

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"DATA_DIR  : {DATA_DIR}")
print(f"Exists    : {DATA_DIR.exists()}")


In [ ]:
import requests

try:
    tags = requests.get("http://localhost:11434/api/tags", timeout=5).json()
    models = [m["name"] for m in tags.get("models", [])]
    print("Ollama running, models:", ", ".join(models) or "(none pulled)")
    assert "qwen2.5-coder:7b" in models, "pull qwen2.5-coder:7b first"
except Exception as exc:
    print(f"Ollama not reachable at localhost:11434 — {exc}")
    print("Install Ollama (ollama.com) and run: ollama pull qwen2.5-coder:7b")


All imports live in one cell: the repo's `evaluation/`, `vectordb/`, and
`retrieval/` modules, plus the libraries they wrap (pdfplumber, Chroma,
Ollama, fastembed, ragas).


In [ ]:
from pathlib import Path
import statistics
import tempfile
import time

import pdfplumber
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter

from evaluation.golden import load_golden
from evaluation.harness import EvaluationHarness
from evaluation.judge import LLMJudge, LocalEmbeddings
from evaluation.metrics import (
    AnswerRelevanceMetric,
    ContextPrecisionMetric,
    ContextRecallMetric,
    FaithfulnessMetric,
)
from evaluation.ragas_metrics import ragas_scores
from retrieval.similarity import SimilarityRetriever
from vectordb.chroma import ChromaVectorStore


In [ ]:
embeddings = LocalEmbeddings()          # fastembed, BAAI/bge-base-en-v1.5
judge = LLMJudge(embedder=embeddings)   # local Ollama judge, temperature 0

stage_times = {}

t0 = time.perf_counter()
vec = embeddings.embed_query("What is the total due?")
print(f"embed dims : {len(vec)}")
print(f"first 5    : {[round(v, 4) for v in vec[:5]]}")
print(f"model load : {time.perf_counter() - t0:.2f}s (cached after first run)")


## 1 · Load — the SD-08 invoice corpus

**WHAT** — extract page text from the six text-based PDF invoices with
pdfplumber.

**WHY** — the corpus is the one from SD-08: same files, same golden set. Two
other PDFs in the folder (`watson-hall-1898.pdf`, `macy-receipt.pdf`) are
scanned images that yield **0 characters** of text, so they are excluded
(OCR for scanned invoices is a later project).

**WHAT TO EXPECT** — a char count per invoice; all six are non-empty.


In [ ]:
DOC_PDFS = [
    "sample-invoice.pdf",
    "multipage_invoice1.pdf",
    "Invoice_1.pdf",
    "Invoice-6.pdf",
    "sdk-invoice1.pdf",
    "german-zugferd.pdf",
]


def extract_pages(path):
    """Return one text string per page via pdfplumber."""
    with pdfplumber.open(path) as pdf:
        return [page.extract_text() or "" for page in pdf.pages]


for name in DOC_PDFS:
    text = "\n".join(extract_pages(DATA_DIR / name)).strip()
    print(f"{name:24} {len(text):6} chars")


## 2 · Split & index

**WHAT** — chunk each invoice with `RecursiveCharacterTextSplitter`
(1000 chars, 100 overlap), tag every chunk with its `invoice_id` metadata,
and persist everything into a fresh Chroma collection.

**WHY** — a fresh `tempfile` directory per run avoids Chroma's documented
accumulation behaviour (re-using a `persist_dir` + `collection_name`
*upserts* instead of starting clean). `invoice_id` metadata is stored so a
metadata-filtered retriever becomes a one-line exercise at the end.

**WHAT TO EXPECT** — a handful of chunks per invoice, then a fast embed + index (local CPU).


In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

all_chunks = []
for name in DOC_PDFS:
    text = "\n".join(extract_pages(DATA_DIR / name))
    doc = Document(
        page_content=text,
        metadata={"invoice_id": Path(name).stem, "source": name},
    )
    chunks = splitter.split_documents([doc])
    all_chunks.extend(chunks)
    print(f"{name:24} -> {len(chunks):3} chunks")

print(f"\nTOTAL chunks: {len(all_chunks)}")


In [ ]:
store = ChromaVectorStore(
    collection_name="p20_deep_eval",
    persist_dir=tempfile.mkdtemp(prefix="p20_chroma_"),
    embedding=embeddings,
)

t0 = time.perf_counter()
store.add(all_chunks)
stage_times["index_build"] = time.perf_counter() - t0
print(f"indexed {len(all_chunks)} chunks in {stage_times['index_build']:.1f}s")


In [ ]:
retriever = SimilarityRetriever(store, top_k=3)

hits = retriever.retrieve("What is the invoice number?")
for d in hits:
    print(f"  [invoice={d.metadata.get('invoice_id')}] {d.page_content[:70]!r}")


## 3 · The golden set

**WHAT** — load the 23 hand-checked question/answer pairs from
`evaluation.golden`.

**WHY** — the golden set is the yardstick. It is extracted **verbatim** from
`NoteBooks/SD-08-Invoices/02-invoice-rag-strategy-comparison.ipynb` (cell 5)
— the same 23 pairs SD-08 used for retrieval recall — so Project 20
measures *generation quality* on exactly the same questions.

**WHAT TO EXPECT** — 23 questions across 6 documents; two example pairs.


In [ ]:
golden = load_golden()
print(f"{len(golden)} questions across {len({g['doc'] for g in golden})} documents")

by_doc = {}
for g in golden:
    by_doc.setdefault(g["doc"], []).append(g)
for doc, items in by_doc.items():
    print(f"  {doc:22} {len(items)} questions")

print("\nFirst two examples:")
for g in golden[:2]:
    print(f"  Q: {g['question']}")
    print(f"  A: {g['reference']}")


## 4 · Retrieval pass — global top-3

**WHAT** — retrieve the top-3 chunks for every golden question and keep them
for the generation and scoring passes.

**WHY** — retrieval here is deliberately **global** (no `invoice_id`
filter): the top-3 comes from *all six invoices*. That is the honest
default, and it exposes a real failure mode — cross-document contamination —
which the next cells make visible.

**WHAT TO EXPECT** — a fast pass (embedding + Chroma only, no LLM calls),
with the top-3 chunk *source invoice* printed for two questions.


In [ ]:
docs_by_q = {}
context_by_q = {}

t0 = time.perf_counter()
for i, g in enumerate(golden, 1):
    docs = retriever.retrieve(g["question"])[:3]
    docs_by_q[g["question"]] = docs
    context_by_q[g["question"]] = "\n\n".join(d.page_content for d in docs)
    if i % 5 == 0:
        print(f"retrieved {i}/{len(golden)}")
stage_times["retrieval"] = time.perf_counter() - t0
print(f"retrieval pass: {stage_times['retrieval']:.1f}s")


Look at the `invoice_id` of each retrieved chunk. `"What is the invoice
number?"` appears in *all six* invoices — so a global top-3 typically mixes
chunks from several documents. The answer the generator produces next can
therefore quote the **wrong invoice's** number. A metadata filter
(`filter={"invoice_id": ...}`, SD-08 nb01) scopes retrieval to one document
and is left as an exercise.


In [ ]:
for q in [
    "What is the invoice number?",
    "What is the total amount on the project statement?",
]:
    print(f"Q: {q}")
    for d in docs_by_q[q]:
        print(f"  [invoice={d.metadata.get('invoice_id')}] {d.page_content[:60]!r}")
    print()


## 5 · Generation pass

**WHAT** — generate one free-form answer per question from its retrieved
context with `LLMJudge.ask` (Ollama `qwen2.5-coder:7b`, temperature 0).

**WHY** — this is the pipeline step the metrics will score. Temperature 0
keeps the run deterministic. Note the judge is told to answer *only from the
context* — so when retrieval is contaminated, the answer inherits the
contamination and the metrics will say so.

**WHAT TO EXPECT** — 23 terse answers, a few minutes of generation, and at
least one visibly wrong answer caused by cross-document contamination.


In [ ]:
answers_by_q = {}

t0 = time.perf_counter()
for i, g in enumerate(golden, 1):
    q = g["question"]
    answers_by_q[q] = judge.ask(q, context_by_q[q])
    if i % 5 == 0:
        print(f"generated {i}/{len(golden)}")
stage_times["generation"] = time.perf_counter() - t0
print(f"generation pass: {stage_times['generation']:.1f}s")


In [ ]:
for q in [
    "What is the invoice number?",
    "What is the total amount on the project statement?",
]:
    print(f"Q: {q}")
    print(f"A: {answers_by_q[q]}\n")


## 6 · From-scratch metrics

**WHAT** — score every (question, context, answer) with the four generation
metrics implemented by hand in `evaluation/metrics.py`:

- **faithfulness** — fraction of the answer's claims supported by the context;
- **answer relevance** — embedding cosine similarity between question and
  answer (terse answers like `"610.00"` score low — expected, discussed at
  the end);
- **context precision** — RAGAS precision@k over the ranked chunks
  (relevant chunks ranked high = high score);
- **context recall** — fraction of the reference answer's claims present in
  the context.

`EvaluationHarness` orchestrates retrieval → generation → scoring.
Because we already retrieved (§4) and generated (§5), we hand the harness a
thin `CachedRetriever` / `CachedJudge` that serve those stored results
instead of paying the LLM twice — the metric scores themselves are still
computed fresh by the real judge.

**WHAT TO EXPECT** — four 0–1 scores per question, an aggregate table, and a
per-question table. This is the longest scoring pass (~5–10 min of judge
calls).


In [ ]:
class CachedRetriever:
    """Return the §4 retrieval so the harness does not re-query Chroma."""

    def __init__(self, cache):
        self.cache = cache

    def retrieve(self, question):
        return self.cache[question]


In [ ]:
class CachedJudge:
    """Serve the §5 answers; delegate judge/embed to the real LLM judge."""

    def __init__(self, judge, answers):
        self._judge = judge
        self._answers = answers

    def ask(self, question, context):
        return self._answers[question]

    def judge(self, instruction, prompt):
        return self._judge.judge(instruction, prompt)

    def embed(self, texts):
        return self._judge.embed(texts)


In [ ]:
faith = FaithfulnessMetric(judge)
rel = AnswerRelevanceMetric(judge)
prec = ContextPrecisionMetric(judge)
rec = ContextRecallMetric(judge)

harness = EvaluationHarness(
    CachedJudge(judge, answers_by_q),
    CachedRetriever(docs_by_q),
    faith,
    rel,
    prec,
    rec,
    top_k=3,
)


In [ ]:
results = []
t0 = time.perf_counter()
for i in range(0, len(golden), 4):
    results.extend(harness.run(golden[i:i + 4]))
    print(f"scored {min(i + 4, len(golden))}/{len(golden)}")
stage_times["from_scratch_metrics"] = time.perf_counter() - t0
print(f"from-scratch metrics: {stage_times['from_scratch_metrics']:.1f}s")


In [ ]:
agg = harness.aggregate(results)

print("Overall — mean over 23 questions")
for name, val in agg["overall"].items():
    print(f"  {name:18} {val:.3f}")

print("\nBy document")
for doc, means in agg["by_doc"].items():
    row = "  ".join(f"{k}={v:.2f}" for k, v in means.items())
    print(f"  {doc:22} {row}")


In [ ]:
harness.print_table(results)


## 7 · RAGAS cross-check

**WHAT** — run the *same* four metrics through the RAGAS 0.4.3 library
(`evaluation/ragas_metrics.py`), using the same local judge and the same
local embeddings.

**WHY** — RAGAS is an independent implementation of these exact metrics. A
side-by-side tells us whether our from-scratch numbers are in the same
ballpark. Be honest about scale: **N = 23 is small** — per-question
prompting differs, so expect noisy deltas rather than identical numbers.

**WHAT TO EXPECT** — the slowest section (~15–20 min: RAGAS answer
relevancy generates extra questions per sample), then a comparison table.


In [ ]:
samples = [
    {
        "user_input": g["question"],
        "response": answers_by_q[g["question"]],
        "retrieved_contexts": [
            d.page_content for d in docs_by_q[g["question"]][:3]
        ],
        "reference": g["reference"],
    }
    for g in golden
]

ragas_llm = ChatOllama(
    model="qwen2.5-coder:7b",
    base_url="http://localhost:11434",
    temperature=0,
)


In [ ]:
ragas_results = []
t0 = time.perf_counter()
for i in range(0, len(samples), 4):
    ragas_results.extend(ragas_scores(samples[i:i + 4], ragas_llm, embeddings))
    print(f"ragas scored {min(i + 4, len(samples))}/{len(samples)}")
stage_times["ragas"] = time.perf_counter() - t0
print(f"ragas: {stage_times['ragas']:.1f}s")


In [ ]:
def mean_non_nan(values):
    vals = [float(v) for v in values if v == v]  # drop NaN
    return sum(vals) / len(vals) if vals else 0.0


ragas_means = {
    m: mean_non_nan(r[m] for r in ragas_results)
    for m in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
}

own = agg["overall"]

pairs = [
    ("faithfulness", "faithfulness"),
    ("answer_relevance", "answer_relevancy"),
    ("context_precision", "context_precision"),
    ("context_recall", "context_recall"),
]
print(f"{'metric':20} {'from-scratch':>12} {'ragas':>12}")
for own_key, ragas_key in pairs:
    print(f"{own_key:20} {own[own_key]:12.3f} {ragas_means[ragas_key]:12.3f}")


## 8 · LLM-as-judge vs a reference-based check

**WHAT** — compute Cohen's kappa between two *labelers* on the same 23
questions:

1. **LLM-as-judge** — faithfulness ≥ 0.5 ⇒ "faithful";
2. **lexical rule** — the reference answer (lowercased) appears as a
   substring of the generated answer.

**WHY** — kappa measures agreement beyond chance (0 = random, 1 = perfect,
> 0.6 ≈ substantial). The lexical rule is a dumb-but-reproducible stand-in
for a second annotator — it tells us how much the judge agrees with a
literal check, and where they diverge (e.g. the judge rewards a
paraphrased number, the substring rule does not).


In [ ]:
labels_judge = [1 if r.faithfulness >= 0.5 else 0 for r in results]
labels_ref = [1 if r.reference.lower() in r.answer.lower() else 0 for r in results]

kappa = EvaluationHarness.kappa(labels_judge, labels_ref)
print("LLM-judge faithfulness (>= 0.5) vs reference-substring rule")
print(f"Cohen's kappa = {kappa:.3f}")

cells = {}
for j, r in zip(labels_judge, labels_ref):
    cells[(j, r)] = cells.get((j, r), 0) + 1
a11, a10 = cells.get((1, 1), 0), cells.get((1, 0), 0)
a01, a00 = cells.get((0, 1), 0), cells.get((0, 0), 0)
print(f"\n{'':8}{'ref=1':>6}{'ref=0':>6}")
print(f"{'judge=1':>8}{a11:>6}{a10:>6}")
print(f"{'judge=0':>8}{a01:>6}{a00:>6}")


## 9 · Latency & cost

**WHAT** — report how long each stage took.

**WHY** — evaluation is a *process*: index → retrieve → generate → score
(twice, from scratch and RAGAS). Knowing where the time goes decides how
you scale up (e.g. drop RAGAS below N=100, cache answers, batch judge
calls). All of it ran locally — the cost column is zero by construction.


In [ ]:
print(f"{'stage':22} {'seconds':>10}")
total = 0.0
for name, secs in stage_times.items():
    print(f"{name:22} {secs:10.1f}")
    total += secs
print(f"{'total':22} {total:10.1f}")

print(
    "\nEstimated API cost: $0.00 — judge (Ollama) and embeddings"
    " (fastembed) ran locally."
)


## What you should notice

- **Contamination is measurable.** Global top-3 retrieval mixes invoices,
  and both the generated answers and the scores show it. This is the
  *point* of Project 20: evaluation turns a vague "answers look wrong" into
  numbers you can act on.
- **Answer relevance is not "correctness".** A terse correct answer like
  `"610.00""` embeds far from the question, so cosine-based relevance scores
  low even when the answer is right — judge relevance together with
  faithfulness, never alone.
- **The judge agrees with a dumb lexical rule at a *moderate* level.**
  Kappa is not 1: the judge catches paraphrased-but-correct answers the
  substring rule misses, and occasionally misses what the rule catches.
- **From-scratch and RAGAS land in the same ballpark, not the same point.**
  At N=23 the prompting differences dominate; the cross-check validates the
  *approach*, it does not promise identical numbers.
- **All of this costs $0.** A local judge + local embeddings are enough for
  serious evaluation on a small golden set.


## Exercises

1. **Scope retrieval per invoice.** Rebuild the retrieval pass with a
   `filter={"invoice_id": <doc>}` on Chroma (SD-08 nb01 pattern) and
   re-run §6. Expect contamination-driven errors (wrong invoice numbers) to
   disappear — watch faithfulness and context recall rise.
2. **Swap the judge model.** Re-run §6 with a different Ollama model (e.g.
   `qwen2.5:7b` or a llama-3.x). Compare kappa and mean scores. Judge
   choice is a hyperparameter.
3. **Add a human labeler.** Print the 23 answers, label them faithful (0/1)
   yourself, and compute kappa between you and the LLM judge.
4. **Scale the golden set.** Add 5–10 QA pairs for the SD-01 Word-document
   corpus and re-run the whole notebook. Watch which metrics are stable at
   larger N.
5. **Threshold sweep.** Move the faithfulness threshold (0.5) and plot kappa
   vs. threshold — find the operating point where judge and rule agree most.
